<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c42-header.png" alt="Nextia Learning · Vision AI: See, Count and Inspect" width="100%">

# Fine-tune a detector on a few images

**[Fine-tune a detector on a few images](https://learning.nextia-ai.com/courses/vision/m04/fine-tune-a-detector-on-a-few-images/)** · Vision AI: See, Count and Inspect · Module 4, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** fine-tune a small detector to find Larkfield's boxes from 40 labelled photos on a CPU, measure it with AP and the count error, and compare it with an open-vocabulary detector that needs no labels.

| | |
|---|---|
| **Time** | About 45 minutes, with about 2 minutes of training |
| **Needs** | An internet connection for the setup cells and the detector's weights (74 MB). No account and no key. |
| **You should know** | Pretrained detectors and AP, from [Pretrained detectors](https://learning.nextia-ai.com/courses/vision/m04/pretrained-detectors/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, Pillow 13.0.0, scikit-learn 1.9.1 and matplotlib 3.11.2 (PyTorch 2.15.0 and timm 1.0.31 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/vision/m04/fine-tune-a-detector-on-a-few-images/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C42/M04-L03-fine-tune-a-detector-on-a-few-images/fine-tune-a-detector-on-a-few-images-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, pillow 13.0.0, scikit-learn 1.9.1, scikit-image 0.26.0, torch 2.15.0, torchvision 0.29.2, timm 1.0.31, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time. PyTorch is large (about 200 MB to download): the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 pillow==13.0.0 scikit-learn==1.9.1 scikit-image==0.26.0 matplotlib==3.11.2 torch==2.15.0 torchvision==0.29.2 timm==1.0.31
import warnings; warnings.filterwarnings("ignore")
import numpy; print("numpy", numpy.__version__)
import pandas; print("pandas", pandas.__version__)
import PIL; print("pillow", PIL.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import skimage; print("scikit-image", skimage.__version__)
import torch; print("torch", torch.__version__)
import torchvision; print("torchvision", torchvision.__version__)
import timm; print("timm", timm.__version__)

## Setup: get the Larkfield images

Run the next cell. It downloads the image sets that this lesson needs (about 6 MB, as zip files) into a folder `larkfield-images`, with the course's helper `compare.py`, checks each file's checksum and unpacks it. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-images` (in Colab: the **Files** panel on the left), then run the cell again.

Where the photos come from: the catalogue photos are cut from Open Images photos (CC BY 2.0: the author of each photo is in `data/catalogue/manifest.csv`), the returned candles are photos of the VisA inspection set (CC BY 4.0), and the pallets are drawn by our code. Read `data/dataset-card.md` for the details.

In [ ]:
# Setup: download the image sets into larkfield-images/, check them and unpack them.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C42/"
FILES = {
    "archives/pallets.zip": "cec4b211fa4e434936fc0c1980accfc74392f1ced9de6c04b5421b9ec6af56e3",
    "archives/extras.zip": "f121137bc765866c25929a4233c557ba72b64eea0c806070cc04a86c28ec27ec",
    "compare.py": "e9794acacbce00c240906cc8b3e2256b8505ff4a8074b7507a8b159cd39b4826",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-images").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-images and run again.")
    if name.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(PROJECT / "data")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
n = sum(1 for p in Path("data").rglob("*") if p.is_file())
print(f"Ready: {n} files in larkfield-images/data.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, Pillow's `Image`, matplotlib, and the course helper as `cp` (the comparison table). It defines `show(paths)`, which draws photos in a row, and the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: images, the comparison table, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import compare as cp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
CLASSES = ["bowl", "candle", "chair", "cushion", "lamp", "mug", "plant_pot", "vase"]

def show(paths, titles=None, size=2.2):
    """Draw photos in a row (at most 8)."""
    paths = list(paths)[:8]
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size))
    for i, (ax, p) in enumerate(zip(np.atleast_1d(axes), paths)):
        ax.imshow(Image.open(p)); ax.axis("off")
        if titles:
            ax.set_title(str(titles[i]), fontsize=8)
    plt.show()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The labelled photos

Each training pallet comes with its true boxes, as a person would label them. We use the 40 training photos and keep the 30 test photos for the end.

In [ ]:
def load(split):
    coco = json.load(open(f"data/pallets/{split}.json"))
    for im in coco["images"]:
        im["boxes"] = [a["bbox"] for a in coco["annotations"] if a["image_id"] == im["id"]]
    return coco["images"]

train, test = load("train"), load("test")
print(len(train), "training photos with", sum(len(im["boxes"]) for im in train), "boxes;", len(test), "test photos")

## 2. Change the head, freeze the backbone

We take the COCO detector of the last lesson and replace its last layer (the **box head**) with a new one for two classes: background and box. The **backbone**, which turns pixels into features, stays as it is (frozen). Only the rest learns.

In [ ]:
import time, torch
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn, FasterRCNN_MobileNet_V3_Large_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms.functional import to_tensor
torch.set_num_threads(4)
torch.manual_seed(0)
model = fasterrcnn_mobilenet_v3_large_fpn(weights=FasterRCNN_MobileNet_V3_Large_FPN_Weights.COCO_V1,
                                          min_size=384, max_size=512, box_detections_per_img=200, box_score_thresh=0.05)
model.roi_heads.box_predictor = FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features, 2)
for p in model.backbone.body.parameters():
    p.requires_grad = False
params = [p for p in model.parameters() if p.requires_grad]
print(f"{sum(p.numel() for p in params):,} of {sum(p.numel() for p in model.parameters()):,} numbers will learn")

## 3. Train for 12 rounds

One **epoch** is one pass over the 40 photos. Each photo is flipped left to right half of the time, so the model sees more variety. `clip_grad_norm_` limits each step: without it, the course's run on 10 photos broke (the loss became `nan`, not a number).

> **Predict.** Will the loss go down in every epoch?

This cell takes about 2 minutes on a laptop CPU.

In [ ]:
opt = torch.optim.SGD(params, lr=0.01, momentum=0.9, weight_decay=1e-4)
data = []
for im in train:
    x = to_tensor(Image.open("data/" + im["file_name"]).convert("RGB"))
    b = torch.tensor([[bx[0], bx[1], bx[0] + bx[2], bx[1] + bx[3]] for bx in im["boxes"]], dtype=torch.float32)
    data.append((x, {"boxes": b, "labels": torch.ones(len(b), dtype=torch.int64)}))
g = torch.Generator().manual_seed(0)
start, losses = time.perf_counter(), []
for epoch in range(12):
    model.train()
    total = 0.0
    for i in torch.randperm(len(data), generator=g).tolist():
        x, target = data[i]
        if torch.rand(1, generator=g).item() < 0.5:  # flip left to right
            x = x.flip(-1)
            bb = target["boxes"].clone()
            bb[:, [0, 2]] = x.shape[-1] - target["boxes"][:, [2, 0]]
            target = {"boxes": bb, "labels": target["labels"]}
        loss = sum(model([x], [target]).values())
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 5.0)
        opt.step()
        total += float(loss)
    losses.append(total / len(data))
    print(f"epoch {epoch + 1:2d}: loss {losses[-1]:.3f}")
print(f"trained in {time.perf_counter() - start:.0f} s")

## 4. Measure on the 30 test photos

In [ ]:
model.eval()
preds = []
for im in test:
    with torch.no_grad():
        o = model([to_tensor(Image.open("data/" + im["file_name"]).convert("RGB"))])[0]
    for b, s in zip(o["boxes"].tolist(), o["scores"].tolist()):
        preds.append({"scene_id": im["scene_id"], "score": s, "box": [b[0], b[1], b[2] - b[0], b[3] - b[1]]})
gts = {im["scene_id"]: im["boxes"] for im in test}
my_ap = cp.average_precision(gts, [(p["scene_id"], p["score"], p["box"]) for p in preds], 0.5)
counts = [sum(1 for p in preds if p["scene_id"] == im["scene_id"] and p["score"] >= 0.6) for im in test]
s = cp.count_scores([im["count"] for im in test], counts, [im["crowding"] for im in test])
print(f"AP50 {my_ap:.3f} | count error at score 0.6: {s['mae']} | by crowding:",
      {k: v["mae"] for k, v in s["by_crowding"].items()})

> **Check.** The course's run gave AP50 0.867 and a count error of 2.5 (tight pallets 6.9). Yours should be close; PyTorch on another CPU can give slightly different numbers.

In [ ]:
expect("your AP50 (about 0.87)", round(my_ap, 2), 0.87, tolerance=0.05)

## 5. No labels at all: an open-vocabulary detector

OWLv2 (Apache-2.0) finds objects from a text, here "a cardboard box", with no training. It is 620 MB and takes about 1.3 seconds a photo on a CPU, so we replay its recorded boxes. The course chose its score line (0.075) and its NMS line (0.3).

In [ ]:
owl = json.load(open("data/predictions/pallets_owlv2_a_cardboard_box.json"))
owl_counts = {}
for im in test:
    P = [p for p in owl if p["scene_id"] == im["scene_id"] and p["score"] >= 0.075]
    owl_counts[im["scene_id"]] = len(cp.nms([p["box"] for p in P], [p["score"] for p in P], 0.3))
for c in ("loose", "medium", "tight"):
    ims = [im for im in test if im["crowding"] == c]
    print(c, "true", [im["count"] for im in ims][:5], "OWLv2", [owl_counts[im["scene_id"]] for im in ims][:5])

## 6. Your turn: OWLv2 on tight pallets

> **Your turn.** Put OWLv2's mean count error on the 10 **tight** pallets in `owl_tight` (mean of the absolute differences between `owl_counts` and the true counts).

In [ ]:
owl_tight = ...

In [ ]:
expect('owl_tight', owl_tight, 36.9, tolerance=0.01)

## 7. Failure case: no labels, no boxes

On tight pallets, OWLv2 finds no box on 8 of the 10 photos, and one box around the whole pallet on the other 2, where there are 22 to 51. Touching boxes look like one big box to it. The fine-tuned detector learned from 40 labelled photos what a single box looks like here. Forty photos with 817 boxes took a person some time to label; the lesson page estimates how long.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/vision/m04/fine-tune-a-detector-on-a-few-images/) has the diagrams, the cost of labels, and the knowledge checks.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Counting, and tracking in brief](https://learning.nextia-ai.com/courses/vision/m04/counting-and-tracking-in-brief/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/vision/m04/fine-tune-a-detector-on-a-few-images/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The photos are open photos (each with its author and licence in the manifests) or images made by our code; no photo shows a real person.